In [ ]:
import os
import requests
from bs4 import BeautifulSoup, Comment
import pandas as pd
import time
from datetime import datetime

/Users/hachikaruanyakwee/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Data directory configuration
DATA_DIR = "data"
RAW_DATA_DIR = os.path.join("..", DATA_DIR, "raw", "serie a brazil")
os.makedirs(RAW_DATA_DIR, exist_ok=True)

In [3]:
# FBref URLs for Brazilian Serie A seasons
SEASONS = {
    "2024": "https://fbref.com/en/comps/24/2024/stats/2024-Serie-A-Stats",
    "2023": "https://fbref.com/en/comps/24/2023/stats/2023-Serie-A-Stats",
    "2022": "https://fbref.com/en/comps/24/2022/stats/2022-Serie-A-Stats",
    "2021": "https://fbref.com/en/comps/24/2021/stats/2021-Serie-A-Stats"
}

In [4]:
def get_fbref_table(url):
    """Extract the stats table from FBref with error handling"""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Find table in comments (FBref's standard approach)
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for comment in comments:
            if 'id="stats_standard"' in str(comment):
                comment_soup = BeautifulSoup(str(comment), "html.parser")
                table = comment_soup.find("table", id="stats_standard")
                if table:
                    return table
        raise ValueError("Stats table not found in page comments")
    except Exception as e:
        raise Exception(f"Failed to extract table: {str(e)}")

In [5]:
def parse_table_to_df(table, season):
    """Convert HTML table to cleaned DataFrame with season metadata"""
    try:
        # Extract headers
        header_rows = table.find("thead").find_all("tr")
        headers = [th.get_text(strip=True) for th in header_rows[-1].find_all("th")]
        
        # Process rows
        rows = table.find("tbody").find_all("tr")
        data = []
        
        for row in rows:
            if row.get("class") and "thead" in row.get("class"):
                continue  # Skip sub-header rows

            cells = row.find_all(["th", "td"])
            row_data = [cell.get_text(strip=True) for cell in cells]
            
            if len(row_data) != len(headers):
                print(f"Skipping malformed row: {row_data[:5]}")
                continue

            data.append(row_data)
        
        if not data:
            raise ValueError("No valid player rows found")
            
        # Create DataFrame
        df = pd.DataFrame(data, columns=headers)
        
        # Clean columns and add metadata
        df.columns = [col.replace("\n", " ").strip() for col in df.columns]
        df['Season'] = season
        df['ScrapeDate'] = datetime.now().strftime('%Y-%m-%d')
        
        return df
        
    except Exception as e:
        raise Exception(f"Table parsing failed: {str(e)}")

In [6]:
def save_data(df, filename):
    """Save DataFrame with validation"""
    try:
        filepath = os.path.join(RAW_DATA_DIR, filename)
        df.to_csv(filepath, index=False, encoding='utf-8-sig')
        print(f"✅ Successfully saved {len(df)} rows to {filepath}")
        return True
    except Exception as e:
        print(f"❌ Failed to save data: {str(e)}")
        return False

In [7]:
# Main function to orchestrate the scraping
def main():
    all_seasons = []
    
    for season, url in SEASONS.items():
        try:
            print(f"\nProcessing {season} season...")
            start_time = time.time()
            
            # Get and parse data
            table = get_fbref_table(url)
            season_df = parse_table_to_df(table, season)
            
            # Save individual season
            if save_data(season_df, f"seriea_{season}_players.csv"):
                all_seasons.append(season_df)
            
            elapsed = time.time() - start_time
            print(f"Completed in {elapsed:.1f} seconds")
            
            # Respectful delay between requests
            time.sleep(25 + abs(hash(season)) % 15)  # 25-40 second delay
            
        except Exception as e:
            print(f"❌ Error processing {season}: {str(e)}")
    
    # Combine and save all seasons
    if all_seasons:
        combined_df = pd.concat(all_seasons, ignore_index=True)
        if save_data(combined_df, "seriea_all_seasons_combined.csv"):
            print("\n=== Combined Data Summary ===")
            print(f"Total players: {len(combined_df)}")
            print("\nSeason distribution:")
            print(combined_df['Season'].value_counts().sort_index(ascending=False))
            
            print("\nSample data:")
            print(combined_df[['Player', 'Pos', 'Squad', 'Season', 'Min', 'Gls']].head())
    else:
        print("\n⚠️ No seasons were successfully processed")

if __name__ == "__main__":
    main()


Processing 2024 season...
✅ Successfully saved 728 rows to ../data/raw/serie a brazil/seriea_2024_players.csv
Completed in 1.2 seconds

Processing 2023 season...
✅ Successfully saved 751 rows to ../data/raw/serie a brazil/seriea_2023_players.csv
Completed in 4.3 seconds

Processing 2022 season...
✅ Successfully saved 761 rows to ../data/raw/serie a brazil/seriea_2022_players.csv
Completed in 4.4 seconds

Processing 2021 season...
✅ Successfully saved 730 rows to ../data/raw/serie a brazil/seriea_2021_players.csv
Completed in 4.9 seconds
✅ Successfully saved 2970 rows to ../data/raw/serie a brazil/seriea_all_seasons_combined.csv

=== Combined Data Summary ===
Total players: 2970

Season distribution:
Season
2024    728
2023    751
2022    761
2021    730
Name: count, dtype: int64

Sample data:
            Player    Pos          Squad Season  Min Gls   Gls
0            Abner     DF      Juventude   2024  108   0  0.00
1  Nicolás Acevedo  MF,DF          Bahia   2024  268   0  0.00
2     